# 01 — Data exploration

First look at the price data for all **10 pairs**, before any signal or backtest logic
touches it. The job here is narrow: confirm the download is complete, the two legs of each
pair line up on the same trading days, and there are no gaps that would silently corrupt a
rolling window downstream.

The ticker list is **not hardcoded** — it comes from `configs/pairs.yaml`, the same file
the backtest reads. A notebook with its own copy of the universe is a notebook that goes
stale the moment a pair is added, which is exactly what happened to the earlier version of
this one.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pairs_teardown.config import load_config
from pairs_teardown.data.clean import align_prices, handle_missing
from pairs_teardown.data.loaders import load_or_download

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "pairs.yaml")

prices_raw = load_or_download(
    list(cfg.tickers), cfg.data.start, cfg.data.end, ROOT / cfg.data.cache_dir
)
pairs = {p.name: align_prices(handle_missing(prices_raw[[p.a, p.b]])) for p in cfg.pairs}

print(f"{len(cfg.pairs)} pairs, {len(cfg.tickers)} tickers, "
      f"{cfg.data.start} to {cfg.data.end}\n")
for p in cfg.pairs:
    print(f"  {p.name:<10} {p.rationale}")


## 1. Coverage

Two things to check, and they are different. **Raw NaNs** tell us whether the download
succeeded. **Post-alignment row counts** tell us how much usable overlapping history each
pair actually has, which is what the backtest sees.


In [ ]:
raw_na = prices_raw.isna().sum()
print(f"raw panel: {len(prices_raw)} rows")
print(f"tickers with missing values: {(raw_na > 0).sum()} of {len(raw_na)}")
if (raw_na > 0).any():
    print(raw_na[raw_na > 0].to_string())


In [ ]:
rows = []
for p in cfg.pairs:
    df = pairs[p.name]
    rows.append({
        "pair": p.name,
        "days": len(df),
        "first": df.index[0].date(),
        "last": df.index[-1].date(),
        "years": round((df.index[-1] - df.index[0]).days / 365.25, 1),
    })
coverage = pd.DataFrame(rows).set_index("pair")
coverage["share of longest"] = (coverage.days / coverage.days.max()).round(3)
coverage


**Nine pairs have the full 2,515 trading days. FOXA/FOX has 1,461.**

That is not a data error. FOX Corporation only began trading on **2019-03-13**, when the
Disney acquisition of Twenty-First Century Fox closed and the remaining business was spun
out with two share classes. There is no earlier price to download.

The consequence is carried through the whole study and is worth stating once here: FOXA/FOX
has 58% of the history of every other pair — and because the missing years are all at the
*start*, its in-sample window is hit hardest: 709 fitting days against 1,763 for everyone
else. Its hedge ratio is fitted on 40% of the data the others get, and every statistic
computed from it is correspondingly noisier. `handle_missing` and `align_prices`
drop the pre-listing rows rather than filling them, because a forward-filled or
zero-filled price would produce a fake stationary spread over a period when one leg did not
exist — the kind of artifact that makes a backtest look better than reality.


## 2. Price levels

Each pair on twin axes. The two legs are on different scales, so the axes are independent —
what matters is whether the *shapes* track each other, which is the visual precondition for
any cointegration story.


In [ ]:
n = len(cfg.pairs)
fig, axes = plt.subplots(n, 1, figsize=(12, 2.8 * n))

for ax, p in zip(axes, cfg.pairs):
    df = pairs[p.name]
    ax2 = ax.twinx()
    ax.plot(df[p.a], color="steelblue", lw=0.9, label=p.a)
    ax2.plot(df[p.b], color="darkorange", lw=0.9, label=p.b)
    ax.set_ylabel(p.a, color="steelblue")
    ax2.set_ylabel(p.b, color="darkorange")
    ax.set_title(f"{p.name} — {p.rationale}", fontsize=10, loc="left")
    ax.legend(loc="upper left", fontsize=8)
    ax2.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()


A visual check is not a test, so here is the number behind it. The log price ratio
`log(A) - log(B)` is the raw material of the spread: if the two legs stay linked it wanders
within a band, and if they come apart it trends. Its total range measures how far apart the
pair ever got.


In [ ]:
import numpy as np

rows = []
for p in cfg.pairs:
    df = pairs[p.name]
    lr = np.log(df[p.a]) - np.log(df[p.b])
    pre, post = lr.loc[:"2021-12-31"], lr.loc["2022-01-01":]
    rows.append({
        "pair": p.name,
        "log-ratio range": lr.max() - lr.min(),
        "drift to 2021": pre.iloc[-1] - pre.iloc[0],
        "drift 2022+": post.iloc[-1] - post.iloc[0],
    })
pd.DataFrame(rows).set_index("pair").sort_values("log-ratio range", ascending=False).round(2)


Two pairs stand out, and both resurface later as the study's worst performers:

- **UPS/FDX** has the widest range of any pair (1.07 in logs, roughly a 3x swing in relative
  value) and it *reverses*: UPS gains 0.39 on FedEx to the end of 2021, then gives back 0.56
  from 2022 as post-pandemic parcel volumes normalized very differently for the two.
- **XOM/CVX** is next (0.82), separating through the 2020–2022 energy cycle and then
  partially reconverging.

At the other extreme, **SPY/VOO's range is 0.01** — the two never meaningfully part, which
is precisely why there is nothing there to trade.

Nothing is excluded on the strength of this. These pairs were specified on economic grounds
and all ten are reported. `02_cointegration_analysis.ipynb` puts formal tests on what the
eye and this table are both picking up.


## 3. Return correlation

Correlation of daily returns is *not* cointegration and is not the thing the strategy
trades — two assets can be highly correlated day to day while their price levels drift
apart forever. It is reported here only as a rough measure of how much common movement
there is to hedge away.


In [ ]:
corr = pd.Series(
    {p.name: pairs[p.name][p.a].pct_change().corr(pairs[p.name][p.b].pct_change())
     for p in cfg.pairs},
    name="daily return correlation",
).sort_values(ascending=False)
corr.round(3).to_frame()


SPY/VOO is essentially 1.00, as it must be — two wrappers on the same index. The rest
sit in a broad 0.4–0.8 band with no obvious grouping.

Keep this table in mind for `03_backtest_results.ipynb`: **SPY/VOO's near-perfect
correlation makes it the study's *worst* risk-adjusted performer**, not its best. Common
movement is what gets hedged out; what is left to trade is the residual, and a pair with
almost no residual has almost nothing to trade while still paying full transaction costs.


## 4. Trading-day alignment

The backtest applies rolling windows over row positions, not calendar dates. If one leg
trades on a day the other does not, an unaligned panel would quietly compare prices from
different days. `align_prices` is what prevents that; this confirms it worked.


In [ ]:
rows = []
for p in cfg.pairs:
    df = pairs[p.name]
    common = prices_raw[[p.a, p.b]].dropna()
    gaps = df.index.to_series().diff().dt.days
    rows.append({
        "pair": p.name,
        "aligned rows": len(df),
        "rows with both legs present": len(common),
        "dropped by alignment": len(common) - len(df),
        "max calendar gap (days)": int(gaps.max()),
        "NaNs remaining": int(df.isna().sum().sum()),
    })
pd.DataFrame(rows).set_index("pair")


No NaNs survive alignment, and nothing is dropped beyond what the raw overlap
already implies. The maximum calendar gap of a few days is just weekends and market
holidays — expected, and harmless because every window in the package is defined over rows
rather than dates.

**Data is clean. Next:** `02_cointegration_analysis.ipynb` asks whether these pairs are
actually cointegrated — and, more importantly, whether that property *survives* into the
out-of-sample period.
